# 06a Symbols, labels and contours

Phase 6a check: circles and lines sized by data, dashed lines, your own labels, the topo theme and contour lines. Contours need `pip install -e .[terrain]` and internet for the elevation tiles. Record pass or fail in the table at the end.

In [1]:
import warnings
import webbrowser
from pathlib import Path

import geopandas as gpd

import kharita as kh

warnings.filterwarnings("ignore", module="lonboard")
states = gpd.read_file("../data/processed/sudan_states.gpkg")
roads = gpd.read_file("../data/processed/khartoum_roads.gpkg")
centres = states.assign(geometry=states.representative_point())
major = roads[roads["rank"] >= 4]
SOURCE = "U.S. Census Bureau via HDX, CC BY 4.0. 2008 census, retabulated."
OVERTURE = "Overture Maps Foundation, OpenStreetMap contributors (ODbL)."
TERRAIN = "Elevation: Mapzen Terrain Tiles on AWS Open Data."

In [2]:
# A. Circles sized by population: area grows with the value; nested-circle key in the legend
kh.map(
    centres,
    "theme-paper size-[pop] size-max-34 fill-rose-500/60 stroke-white stroke-1 tooltip-[key,pop]",
    title="Population by state",
    source=SOURCE,
)

In [3]:
# B. Two layers: density as colour, population as circles, state codes as labels
m = kh.Map(title="Density and population", source=SOURCE)
m.add(states, "theme-paper fill-[density] ramp-ylorrd scheme-q5 stroke-white label-[key]", legend={"title": "People per km2", "style": "bar"})
m.add(centres, "size-[pop] size-max-30 fill-slate-900/30 stroke-slate-900 stroke-1")
m

In [4]:
# C. Line width by road rank, colour by class, on midnight
kh.map(
    roads,
    "theme-midnight stroke-[road_class] ramp-set2 width-[rank] width-max-5 tooltip-[name,road_class]",
    legend={"title": "Road class", "note": OVERTURE},
    title="Central Khartoum roads",
)

In [5]:
# D. Dashed outlines, the way state borders are drawn on official maps
kh.map(states, "basemap-paper fill-transparent stroke-slate-700 stroke-1.5 dash-4-2", title="Dashed state borders")

In [6]:
# E. Labels along lines: major road names follow the roads
kh.map(
    major,
    "basemap-muted stroke-rose-500 width-[rank] width-max-4 label-[name] legend-none",
    title="Road names along the roads",
    source=OVERTURE,
)

In [12]:
# F. Topographic map of Jebel Marra: contours every 100 m, index contours thicker and labelled
marra = kh.contours((24.0, 12.75, 24.5, 13.15), interval=100)
topo = kh.Map(title="Jebel Marra", subtitle="Contours every 100 m", source=TERRAIN)
topo.add(marra, "preset-topo basemap-topo")
topo.add(marra[marra["index"]], "stroke-transparent label-[elevation] legend-none")
topo

In [8]:
# G. Contours coloured by height over sharp relief, on midnight
kh.map(
    marra,
    "basemap-midnight relief-sharp stroke-[elevation] ramp-thermal scheme-linear width-[weight] width-max-1.8",
    legend={"title": "Elevation, m", "fmt": "{:,.0f}"},
    title="Jebel Marra by height",
)

In [9]:
# H. A bigger area chooses its own interval and zoom: the Red Sea hills
hills = kh.contours((36.0, 18.6, 37.4, 19.8))
print(len(hills), "lines, interval", hills.attrs["interval"], "m, zoom", hills.attrs["zoom"])
kh.map(hills, "preset-topo basemap-topo relief-steps", title="Red Sea hills", source=TERRAIN)

662 lines, interval 100 m, zoom 9


In [10]:
# I. Export the topographic map
out = Path("../spike_out")
out.mkdir(exist_ok=True)
topo.to_html(out / "jebel_marra_topo.html")
webbrowser.open((out / "jebel_marra_topo.html").resolve().as_uri())

True

## Results

| Test | What passing looks like | Pass? | Notes |
| --- | --- | --- | --- |
| A | Circles grow with population; legend shows three nested circles | yes | it works |
| B | Coloured states, circles on top, state codes as labels, no overlaps | yes | it works |
| C | Major roads thick, minor thin, eight colours; width key in the legend | yes | it works |
| D | State borders dashed | yes | it works |
| E | Road names follow the roads | yes | it works |
| F | Brown contours, thicker index lines with height labels, like your reference image | yes | but why is it not as clean as the ones i found in the internet its all over the place with alot of small details being taken into considrations i think we need to modify this to be better and also for the next test of this add 2 maps view the current one and also the satalite imagery if that is possible or combination of the old view we used in pahase 5 plus this view i want to see if i can compine map view with the topography |
| G | Contours from dark to bright by height, over relief | yes | the color are working according to the high but the same issue with the f there is too match details for the circles we created we need to improve on this |
| H | Contours across the hills over stepped relief; interval and zoom printed | yes | i think now that maybe its sudan land that is complex for a tropography let me know |
| I | Exported page matches F | yes | it works |